In [1]:
from tqdm import tqdm
import os
from os import listdir
import time
from random import randint
from os.path import isfile, join
 
import gc 
import numpy as np
from scipy import stats
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import KFold

import nibabel as nib
import pydicom as pdm
import nilearn as nl
import nilearn.plotting as nlplt
import h5py

import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.animation as anim
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

import seaborn as sns
import imageio
from skimage.transform import resize
from skimage.util import montage

# from IPython.display import Image as show_gif
# from IPython.display import clear_output
# from IPython.display import YouTubeVideo

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn import MSELoss

# !pip install opencv-python==4.6.0.66
# !pip install -U albumentations --no-binary qudida,albumentations
import albumentations as A
# from albumentations.pytorch import ToTensor, ToTensorV2


from albumentations import Compose, HorizontalFlip
# from albumentations.pytorch import ToTensor, ToTensorV2 

import warnings
warnings.simplefilter("ignore")

# Function to Calculate Volume of a Tumor based on mask file

In [50]:
def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET, mask_ET])
    
    return mask 

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = 'BraTS-GLI-' + patient_id + '/' + 'BraTS-GLI-' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']

    sample_filename1 = baseloc + pefix + suffixs[0]
    sample_img1_f = nib.load(sample_filename1)
    sample_img1 = np.asarray(sample_img1_f.dataobj)
    # sample_img1 = np.rot90(sample_img1)

    sample_filename2 = baseloc + pefix + suffixs[1]
    sample_img2_f = nib.load(sample_filename2)
    sample_img2 = np.asarray(sample_img2_f.dataobj)
    # sample_img2  = np.rot90(sample_img2)

    sample_filename3 = baseloc + pefix + suffixs[2]
    sample_img3_f = nib.load(sample_filename3)
    sample_img3 = np.asarray(sample_img3_f.dataobj)
    # sample_img3  = np.rot90(sample_img3)

    sample_filename4 = baseloc + pefix + suffixs[3]
    sample_img4_f = nib.load(sample_filename4)
    sample_img4 = np.asarray(sample_img4_f.dataobj)
    # sample_img4  = np.rot90(sample_img4)

    sample_filename_mask = baseloc + pefix + suffixs[4]
    sample_mask_f = nib.load(sample_filename_mask)
    sample_mask = np.asarray(sample_mask_f.dataobj)
    
    return sample_img1, sample_img2, sample_img3, sample_img4, sample_mask 

def calculate_tumor_position(dataset, patient_id):
    sample_img1, sample_img2, sample_img3, sample_img4, mask  = read_MRI(dataset, patient_id)
    
    processed_mask = preprocess_mask_labels(mask)

    # Assuming mri_mask is a 3D numpy array with tumor labels 1, 2, 3 and background 0

    # Calculate the center of the MRI scan
    center = np.array(processed_mask[0].shape) / 2

    # Prepare a dictionary to store tumor positions
    tumor_positions = {}
    
    tumor_subsections = {0:'WT', 1:'TC', 2:'ET'}

    # Iterate through each tumor label
    for _subsections in [0,1,2]:
        sample_mask = processed_mask[_subsections]
        
        # Find coordinates of the current tumor label
        tumor_coords = np.argwhere(sample_mask == 1)
        
        # If there are no pixels with the current label, skip to the next label
        if tumor_coords.size == 0:
            tumor_positions[f"{tumor_subsections[_subsections]}"] = {'Position': np.nan, 'Distance': np.nan}
            continue

        # Calculate the centroid of the tumor
        centroid = np.mean(tumor_coords, axis=0)

        # Calculate the position relative to the center of the scan
        position = centroid - center
        
        # Calculate the Euclidean distance from the center
        distance = np.linalg.norm(position)

        # Store the position in the dictionary
        tumor_positions[f"{tumor_subsections[_subsections]}"] = {'Position': position, 'Distance': distance}

    return tumor_positions

# Calculate the Tumor Volume 

In [51]:
dataset = 'Brats2023'
data_path = '../input/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
patient_ids = [f for f in listdir(data_path) if not isfile(join(data_path, f))]

tumor_position = {}

for _id in patient_ids:
    patient_id = _id.split('GLI-')[1]
    tumor_positions = calculate_tumor_position(dataset, patient_id)
    
    tumor_position[_id] = [tumor_positions['WT']['Distance'], 
                           tumor_positions['TC']['Distance'], 
                           tumor_positions['ET']['Distance']]
tumor_position_df = pd.DataFrame.from_dict(tumor_position, 
                                         orient = 'index', 
                                         columns = ['WT_position', 
                                                    'TC_position', 
                                                    'ET_position'])

tumor_position_df.to_csv('../input/BraTS2023/GLI-Tumor_position.csv')

NameError: name 'tumor_volume_df' is not defined